# Vision Transformer (ViT)

**Paper:** A. Dosovitskiy, L. Beyer, A. Kolesnikov, D. Weissenborn, X. Zhai, T. Unterthiner, M. Dehghani, M. Minderer, G. Heigold, S. Gelly, J. Uszkoreit, and N. Houlsby, "An Image is Worth 16×16 Words: Transformers for Image Recognition at Scale," in *International Conference on Learning Representations (ICLR)*, 2021. [arXiv:2010.11929](https://arxiv.org/abs/2010.11929)

**Summary:** ViT applies a pure Transformer encoder to sequences of fixed-size image patches with minimal inductive bias, showing that large-scale pre-training enables transformers to match or exceed CNN performance on image recognition.

## Architecture Overview

```
Input Image [B, 3, H, W]
        │
        ▼  PatchEmbed (Conv2d, kernel=patch_size, stride=patch_size)
Patches [B, N, D]   where N = (H/p)*(W/p)
        │
        ▼  Prepend CLS token → cat([cls, patches]) → shape [B, N+1, D]
        │  Add learned 1D positional embedding (one vector per token)
        │
        ▼  ┌─────────────────────────────────────┐
           │  ViTBlock × depth                   │
           │  ┌────────────────────────────────┐ │
           │  │  x = x + Attn(LayerNorm(x))   │ │  ← Pre-norm
           │  │  x = x + MLP(LayerNorm(x))    │ │
           │  │  Attn: Q,K,V via single in_proj│ │
           │  │  MLP:  Linear→GELU→Linear 4×  │ │
           │  └────────────────────────────────┘ │
           └─────────────────────────────────────┘
        │
        ▼  Final LayerNorm
        │
        ├──► pooled_output = x[:, 0]       [B, D]      (CLS token)
        └──► patch_tokens  = x[:, 1:]      [B, N, D]
             feature_maps  = reshape(patch_tokens) → [B, D, H/p, W/p]
```

### Key Design Decisions
| Choice | Value | Reason |
|--------|-------|--------|
| Pre-norm | LN before residual branch | More stable than post-norm at scale |
| Positional encoding | Learned 1D absolute | Simple; 2D or RoPE used by later models |
| Pooling | CLS token | Dedicated summary token; patch tokens remain spatial |
| MLP activation | GELU | Smoother than ReLU; marginal gains at scale |
| QKV projection | Single `in_proj` (3D→3×D) | Cache-friendly, matches NLP Transformer |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.patch.set_facecolor('#0f1117')

# ── Left: ViT Architecture Diagram ──────────────────────────────────────────
ax = axes[0]
ax.set_facecolor('#0f1117')
ax.set_xlim(0, 10)
ax.set_ylim(0, 12)
ax.axis('off')
ax.set_title('ViT Architecture', color='white', fontsize=14, fontweight='bold', pad=10)

BOX_CLR = {'embed': '#1a3a5c', 'block': '#1a4a3a', 'cls': '#4a1a3a',
           'pos': '#3a3a1a', 'norm': '#2a2a4a', 'out': '#1a4a4a'}
TXT = 'white'

def draw_box(ax, x, y, w, h, label, sublabel='', color='#1a3a5c', fontsize=9):
    box = FancyBboxPatch((x - w/2, y - h/2), w, h,
                          boxstyle='round,pad=0.05', facecolor=color,
                          edgecolor='#4a9eff', linewidth=1.5)
    ax.add_patch(box)
    ax.text(x, y + (0.15 if sublabel else 0), label, ha='center', va='center',
            color=TXT, fontsize=fontsize, fontweight='bold')
    if sublabel:
        ax.text(x, y - 0.22, sublabel, ha='center', va='center',
                color='#aaaaaa', fontsize=7)

def arrow(ax, x, y1, y2, color='#4a9eff'):
    ax.annotate('', xy=(x, y2 + 0.05), xytext=(x, y1 - 0.05),
                arrowprops=dict(arrowstyle='->', color=color, lw=1.5))

# Image patches
for i, (xi, yi) in enumerate([(1.5+j*0.55, 11) for j in range(5)]):
    c = FancyBboxPatch((xi-0.22, yi-0.22), 0.44, 0.44,
                        boxstyle='square,pad=0', facecolor=plt.cm.viridis(i/5),
                        edgecolor='white', linewidth=0.8)
    ax.add_patch(c)
ax.text(5, 11, '... N patches', ha='center', va='center', color='#aaaaaa', fontsize=8)
ax.text(5, 11.7, 'Image [B, 3, 224, 224]', ha='center', va='center', color=TXT, fontsize=9, style='italic')

arrow(ax, 5, 10.55, 10.15)
draw_box(ax, 5, 9.9, 6.5, 0.6, 'PatchEmbed', 'Conv2d(3→D, kernel=patch_size, stride=patch_size)', BOX_CLR['embed'])

arrow(ax, 5, 9.6, 9.2)
draw_box(ax, 5, 8.95, 6.5, 0.55, '[CLS] token prepend + pos_embed add', '', BOX_CLR['pos'])
ax.text(5, 8.6, 'Sequence: [B, N+1, D]  (N = H·W / p²)', ha='center', va='center',
        color='#aaaaaa', fontsize=7.5)

arrow(ax, 5, 8.3, 7.9)
# Stacked ViT blocks
for i in range(3):
    y_blk = 7.6 - i * 1.15
    draw_box(ax, 5, y_blk, 6.5, 0.85,
             f'ViTBlock {i+1}' + (' ... ×depth' if i == 2 else ''),
             'LN→MHSA→Add  +  LN→MLP(4×)→Add', BOX_CLR['block'], fontsize=8.5)
    if i < 2:
        arrow(ax, 5, y_blk - 0.47, y_blk - 0.78)

arrow(ax, 5, 4.75, 4.35)
draw_box(ax, 5, 4.1, 6.5, 0.55, 'Final LayerNorm', '', BOX_CLR['norm'])

arrow(ax, 5, 3.82, 3.4)
# Two outputs
draw_box(ax, 2.5, 2.9, 2.8, 0.65, 'pooled_output', 'x[:,0]  [B, D]', BOX_CLR['out'])
draw_box(ax, 7.5, 2.9, 2.8, 0.65, 'patch_tokens', 'x[:,1:]  [B, N, D]', BOX_CLR['cls'])
ax.annotate('', xy=(2.5, 3.22), xytext=(5, 3.62),
            arrowprops=dict(arrowstyle='->', color='#ff9944', lw=1.5))
ax.annotate('', xy=(7.5, 3.22), xytext=(5, 3.62),
            arrowprops=dict(arrowstyle='->', color='#44ff99', lw=1.5))

# ── Right: Attention heatmap visualization ───────────────────────────────────
ax2 = axes[1]
ax2.set_facecolor('#0f1117')
ax2.set_title('Simulated CLS Attention Map (Head 0, Layer 11)', color='white', fontsize=12)

# Generate a plausible-looking attention map: high attention on object, low on bg
rng = np.random.default_rng(42)
grid = 14  # 224/16
attn_map = rng.random((grid, grid)) * 0.1
# Create a circular object region with high attention
yy, xx = np.ogrid[:grid, :grid]
mask = (xx - 7)**2 + (yy - 5)**2 < 18
attn_map[mask] += rng.random(mask.sum()) * 0.6 + 0.3
attn_map /= attn_map.sum()
attn_map = attn_map / attn_map.max()

im = ax2.imshow(attn_map, cmap='inferno', interpolation='nearest', vmin=0, vmax=1)
plt.colorbar(im, ax=ax2, fraction=0.046, pad=0.04, label='Attention weight')
ax2.set_xlabel('Patch column', color='white')
ax2.set_ylabel('Patch row', color='white')
ax2.tick_params(colors='white')
for spine in ax2.spines.values():
    spine.set_edgecolor('#4a9eff')

ax2.text(0.5, -0.12,
         'CLS token attends strongly to the object (warm) and ignores background (cold)',
         transform=ax2.transAxes, ha='center', color='#aaaaaa', fontsize=9)

plt.tight_layout()
plt.savefig('vit_architecture.png', dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()
print('Architecture diagram saved.')

## Key Innovations

### 1. Pure Transformer for Vision
Prior work (e.g., BERT for NLP) showed transformers excel at sequences. ViT asks: *can we apply the same architecture to images with minimal modification?*

The answer: **yes**, if pre-trained at sufficient scale (JFT-300M, ImageNet-21k). At smaller scales (ImageNet-1k alone), CNNs still have an advantage due to their translation equivariance inductive bias.

### 2. Scaled Dot-Product Attention

$$\text{Attn}(Q, K, V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_{head}}}\right)V$$

- $Q, K, V \in \mathbb{R}^{B \times H \times N \times d_{head}}$  
- The $\sqrt{d_{head}}$ scale prevents softmax saturation in high-dim spaces
- All tokens attend to all other tokens: $O(N^2)$ complexity in sequence length

### 3. Combined QKV Projection
Instead of three separate linear layers:
$$[Q, K, V] = \text{split}\bigl(x \cdot W_{in\_proj}^\top\bigr), \quad W_{in\_proj} \in \mathbb{R}^{3D \times D}$$
This is cache-friendly because a single GEMM replaces three smaller ones.

### 4. CLS Token Pooling
A learnable `[CLS]` token is prepended to the patch sequence. After L transformer blocks, the CLS token integrates information from all patches via attention. Its final embedding becomes the image representation:
$$z_{pooled} = \text{LN}(x)_{[:, 0]}$$

### 5. Model Variants Summary
| Name | D | Heads | Layers | Patch | Params |
|------|---|-------|--------|-------|--------|
| ViT-Ti/16 | 192 | 3 | 12 | 16 | ~5.7M |
| ViT-S/16 | 384 | 6 | 12 | 16 | ~22M |
| ViT-B/16 | 768 | 12 | 12 | 16 | ~86M |
| ViT-L/16 | 1024 | 16 | 24 | 16 | ~307M |
| ViT-H/14 | 1280 | 16 | 32 | 14 | ~632M |
| ViT-22B/14 | 6144 | 48 | 48 | 14 | ~22B |

## Module Walkthrough

### PatchEmbed — Image → Token Sequence
```python
class PatchEmbed(nn.Module):
    def __init__(self, img_size=224, patch_size=16, in_chans=3, embed_dim=768):
        self.proj = nn.Conv2d(in_chans, embed_dim,
                              kernel_size=patch_size, stride=patch_size)
        # Equivalent to: flatten each patch and apply a shared linear layer
        # But Conv2d is faster due to optimised BLAS layout

    def forward(self, x):           # [B, 3, H, W]
        x = self.proj(x)            # [B, D, H/p, W/p]
        x = x.flatten(2).T(1,2)    # [B, N, D]  (N = (H/p)²)
        return x
```

### ViTBlock — Pre-Norm Transformer Block
```python
class ViTBlock(nn.Module):
    def forward(self, x, return_attn=False):
        # Pre-norm: LayerNorm BEFORE the attention/MLP branch
        attn_out, attn_weights = self.attn(self.norm1(x), return_attn=return_attn)
        x = x + attn_out                    # residual connection 1
        x = x + self.mlp(self.norm2(x))     # residual connection 2
        return x, attn_weights
```

### ViTAttention — Combined QKV with Scaled Dot-Product
```python
class ViTAttention(nn.Module):
    def forward(self, x, return_attn=False):
        B, N, D = x.shape
        qkv = self.in_proj(x)                             # [B, N, 3D]
        qkv = qkv.reshape(B, N, 3, H, d).permute(2,0,3,1,4)
        q, k, v = qkv.unbind(0)                           # each [B, H, N, d]
        attn = (q @ k.T(-2,-1)) * self.scale              # [B, H, N, N]
        attn = softmax(attn, dim=-1)
        out  = (attn @ v).T(1,2).reshape(B, N, D)
        return self.out_proj(out), attn
```

### ViTBackbone — Forward Pass
```python
def forward(self, x):
    B = x.shape[0]
    x = self.patch_embed(x)                     # [B, N, D]
    cls = self.cls_token.expand(B, -1, -1)      # [B, 1, D]
    x = torch.cat([cls, x], dim=1)              # [B, N+1, D]
    x = self.pos_drop(x + self.pos_embed)       # add position info
    for blk in self.blocks:
        x, attn = blk(x)                        # L × transformer blocks
    x = self.norm(x)
    pooled = x[:, 0]                            # CLS token → global repr
    patches = x[:, 1:]                          # patch tokens
    feat_map = patches.T(1,2).reshape(B,D,G,G)  # spatial grid
    return BackboneOutput(pooled, patches, [feat_map])
```

In [ ]:
import sys
sys.path.insert(0, '../../src')

import torch
from hale_core.nn.vision import build_vision_backbone

# Build a ViT-Small and run a forward pass
backbone = build_vision_backbone('vit_small', img_size=224, return_all_attn=True)
backbone.eval()
print(f'Parameters: {backbone.num_parameters():,}')
print(f'Embed dim : {backbone.embed_dim}')

x = torch.randn(1, 3, 224, 224)
with torch.no_grad():
    out = backbone(x)

print(f'pooled_output : {out.pooled_output.shape}')    # [1, 384]
print(f'patch_tokens  : {out.patch_tokens.shape}')     # [1, 196, 384]
print(f'feature_maps  : {out.feature_maps[0].shape}')  # [1, 384, 14, 14]
print(f'Attention maps: {len(out.attentions)} layers')
print(f'  Layer 0 attn: {out.attentions[0].shape}')    # [1, 6, 197, 197]

# Visualise attention from CLS token to patches (head 0, last layer)
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
fig.patch.set_facecolor('#0f1117')
fig.suptitle('ViT-S CLS Token Attention — Last Layer, All 6 Heads', color='white', fontsize=13)

last_attn = out.attentions[-1][0]  # [H, N+1, N+1]
cls_attn = last_attn[:, 0, 1:]     # CLS→patches: [H, N]
cls_attn = cls_attn.reshape(-1, 14, 14).numpy()

for i, ax in enumerate(axes.flat):
    ax.set_facecolor('#0f1117')
    if i < 6:
        im = ax.imshow(cls_attn[i], cmap='magma', interpolation='nearest')
        ax.set_title(f'Head {i}', color='white', fontsize=10)
        plt.colorbar(im, ax=ax, fraction=0.046)
    ax.tick_params(colors='white')
    for sp in ax.spines.values():
        sp.set_edgecolor('#333')

plt.tight_layout()
plt.savefig('vit_attn_heads.png', dpi=120, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()
print('\nDifferent heads specialise in different spatial regions of the image.')

In [ ]:
# ── Variant comparison chart ─────────────────────────────────────────────────
import matplotlib.ticker as mticker

variants = {
    'Ti/16': {'params_M': 5.7,  'dim': 192,  'heads': 3,  'depth': 12},
    'S/16':  {'params_M': 22,   'dim': 384,  'heads': 6,  'depth': 12},
    'B/16':  {'params_M': 86,   'dim': 768,  'heads': 12, 'depth': 12},
    'L/16':  {'params_M': 307,  'dim': 1024, 'heads': 16, 'depth': 24},
    'H/14':  {'params_M': 632,  'dim': 1280, 'heads': 16, 'depth': 32},
}

names = list(variants.keys())
params = [v['params_M'] for v in variants.values()]
dims   = [v['dim']      for v in variants.values()]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
fig.patch.set_facecolor('#0f1117')
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']

for ax in [ax1, ax2]:
    ax.set_facecolor('#1a1a2e')
    ax.tick_params(colors='white')
    for sp in ax.spines.values():
        sp.set_edgecolor('#444')

ax1.bar(names, params, color=colors, edgecolor='#555', linewidth=0.8)
ax1.set_title('Parameter Count (M)', color='white', fontsize=12)
ax1.set_ylabel('Millions', color='white')
ax1.set_yscale('log')
for i, (n, p) in enumerate(zip(names, params)):
    ax1.text(i, p * 1.3, f'{p}M', ha='center', color='white', fontsize=9)

ax2.bar(names, dims, color=colors, edgecolor='#555', linewidth=0.8)
ax2.set_title('Embedding Dimension D', color='white', fontsize=12)
ax2.set_ylabel('Dimension', color='white')
for i, (n, d) in enumerate(zip(names, dims)):
    ax2.text(i, d + 30, str(d), ha='center', color='white', fontsize=9)

fig.suptitle('ViT Variant Comparison', color='white', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('vit_variants.png', dpi=120, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()

## References

### Primary
1. A. Dosovitskiy et al., "An Image is Worth 16×16 Words: Transformers for Image Recognition at Scale," *ICLR 2021*. [arXiv:2010.11929](https://arxiv.org/abs/2010.11929)
2. X. Zhai et al., "Scaling Vision Transformers to 22 Billion Parameters," *ICML 2022*. [arXiv:2302.05442](https://arxiv.org/abs/2302.05442)

### Related Works
3. A. Vaswani et al., "Attention Is All You Need," *NeurIPS 2017*. [arXiv:1706.03762](https://arxiv.org/abs/1706.03762) — Original Transformer
4. J. Devlin et al., "BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding," *NAACL 2019*. [arXiv:1810.04805](https://arxiv.org/abs/1810.04805)
5. H. Touvron et al., "Training data-efficient image transformers & distillation through attention," *ICML 2021*. [arXiv:2012.12877](https://arxiv.org/abs/2012.12877) — DeiT
6. Z. Liu et al., "Swin Transformer: Hierarchical Vision Transformer using Shifted Windows," *ICCV 2021*. [arXiv:2103.14030](https://arxiv.org/abs/2103.14030)
7. K. He et al., "Masked Autoencoders Are Scalable Vision Learners," *CVPR 2022*. [arXiv:2111.06377](https://arxiv.org/abs/2111.06377) — MAE pre-training for ViT